# Project 6 — Fine-Tuned Classifier (Starter)

**Goal:** Fine-tune a small model on a classification task and compare it against a prompt-only baseline.

**Stack:** HuggingFace Transformers · PEFT · TRL · Datasets · OpenAI (for baseline)

**Exercise cells are marked with `# YOUR CODE HERE`.**

> **GPU required** for actual training. The evaluation and comparison cells work on CPU.

---

**What you'll build:**
1. Prepare a classification dataset in chat format
2. Establish a prompt-only baseline with GPT-4o-mini
3. Fine-tune with QLoRA
4. Compare fine-tuned vs baseline on a held-out test set

In [ ]:
%pip install -q transformers peft trl bitsandbytes datasets accelerate openai scikit-learn

In [ ]:
# Setup
import os, json
import torch
from datasets import Dataset
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Customer support ticket classification dataset
RAW_DATA = [
    {"text": "My order hasn't arrived yet and it's been 2 weeks!", "label": "shipping"},
    {"text": "I was charged twice for the same order.", "label": "billing"},
    {"text": "The product stopped working after 3 days.", "label": "defect"},
    {"text": "How do I return an item I don't want?", "label": "returns"},
    {"text": "Can I change my delivery address?", "label": "shipping"},
    {"text": "I want a refund for my purchase.", "label": "billing"},
    {"text": "The screen is cracked and I haven't dropped it.", "label": "defect"},
    {"text": "What is your return window?", "label": "returns"},
    {"text": "My package shows delivered but I haven't received it.", "label": "shipping"},
    {"text": "I was charged the wrong amount.", "label": "billing"},
    {"text": "The battery drains in 2 hours instead of 24.", "label": "defect"},
    {"text": "Do I need the original packaging to return?", "label": "returns"},
    {"text": "Tracking says my order is stuck in transit.", "label": "shipping"},
    {"text": "I was charged a cancellation fee I didn't agree to.", "label": "billing"},
    {"text": "The zipper broke after one use.", "label": "defect"},
    {"text": "Can I exchange for a different size?", "label": "returns"},
]

LABELS = ["shipping", "billing", "defect", "returns"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

print(f"Dataset: {len(RAW_DATA)} examples, {len(LABELS)} classes")
print(f"Classes: {LABELS}")

## Step 1 · Baseline: Prompt-Only Classification

In [ ]:
# Exercise 1: Implement a zero-shot baseline using GPT-4o-mini
# The function should classify a support ticket into one of the 4 categories
# Return only the label (lowercase string)

def classify_baseline(text: str) -> str:
    # YOUR CODE HERE
    # System prompt: explain the 4 categories
    # User prompt: the ticket text
    # Return one of: "shipping", "billing", "defect", "returns"
    return ""

# Quick test
test_ticket = "My package tracking hasn't updated in 5 days."
prediction = classify_baseline(test_ticket)
print(f"Text: {test_ticket}")
print(f"Predicted: {prediction}")

In [ ]:
# Exercise 2: Evaluate the baseline on all test examples
# Split data: 12 train, 4 test (use the last 4 examples as test set)

train_data = RAW_DATA[:12]
test_data = RAW_DATA[12:]

def evaluate_classifier(classify_fn, test_set: list) -> dict:
    # YOUR CODE HERE
    # Run classify_fn on each test example
    # Return {"accuracy": float, "predictions": list, "actuals": list}
    return {"accuracy": 0.0, "predictions": [], "actuals": []}

baseline_results = evaluate_classifier(classify_baseline, test_data)
print(f"Baseline accuracy: {baseline_results['accuracy']:.1%}")
print()
for pred, actual, ex in zip(baseline_results["predictions"], baseline_results["actuals"], test_data):
    status = "✓" if pred == actual else "✗"
    print(f"  {status} Actual: {actual:<10} Predicted: {pred:<10} — {ex['text'][:50]}")

## Step 2 · Prepare Training Data

In [ ]:
# Exercise 3: Format training data in chat template format
# Each example should have:
# - System: "Classify this customer support ticket. Reply with one word: shipping, billing, defect, or returns."
# - User: the ticket text
# - Assistant: the label
# Format: {"text": "<|system|>...\n<|user|>...\n<|assistant|>..."}

SYSTEM_PROMPT = "Classify this customer support ticket. Reply with exactly one word: shipping, billing, defect, or returns."

def format_for_training(example: dict) -> dict:
    # YOUR CODE HERE
    return {"text": ""}

formatted_train = [format_for_training(ex) for ex in train_data]
train_dataset = Dataset.from_list(formatted_train)

print("Sample training record:")
print(train_dataset[0]["text"] if train_dataset else "Not implemented")

## Step 3 · QLoRA Fine-Tuning

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # small enough for free Colab T4
OUTPUT_DIR = "./finetuned-classifier"

if torch.cuda.is_available():
    # Exercise 4: Complete the QLoRA setup
    # Configure BitsAndBytesConfig (nf4, 4-bit)
    # Load model with quantization
    # Configure LoRA (r=8, alpha=16 for small dataset)
    # Set up SFTTrainer with 5 epochs, batch_size=4

    # YOUR CODE HERE
    print("Fine-tuning setup not yet implemented")
    # trainer.train()  # uncomment to train
else:
    print("No GPU — skipping fine-tuning setup")
    print("Expected config:")
    print("  Model: TinyLlama-1.1B")
    print("  LoRA: r=8, alpha=16")
    print("  Epochs: 5, batch=4")

## Step 4 · Inference with Fine-Tuned Model

In [ ]:
# Exercise 5: Load the fine-tuned model and implement classify_finetuned()
# Use the same format as the training data
# Return only the label (strip and lowercase the first word of the output)

def classify_finetuned(text: str, model=None, tokenizer=None) -> str:
    if model is None or tokenizer is None:
        return "model_not_loaded"
    # YOUR CODE HERE
    # Format prompt → tokenize → generate (max 5 tokens, temperature=0)
    # Extract and return the label
    return ""

# After training, load and evaluate:
# from peft import PeftModel
# base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, ...)
# ft_model = PeftModel.from_pretrained(base, OUTPUT_DIR)
# ft_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
# classify_fn = lambda text: classify_finetuned(text, ft_model, ft_tokenizer)
# ft_results = evaluate_classifier(classify_fn, test_data)

print("Load fine-tuned model after training is complete")

## Step 5 · Compare Baseline vs Fine-Tuned

In [ ]:
# Exercise 6: Create a comparison report
# Show: accuracy, per-class F1, latency, cost per 1000 classifications
# Decide: when would fine-tuning be worth it for this task?

from sklearn.metrics import classification_report

def comparison_report(baseline_results: dict, finetuned_results: dict = None) -> None:
    print("=" * 50)
    print("BASELINE (gpt-4o-mini, zero-shot)")
    print(f"  Accuracy: {baseline_results['accuracy']:.1%}")
    if baseline_results["predictions"]:
        print(classification_report(
            baseline_results["actuals"],
            baseline_results["predictions"],
            labels=LABELS,
            target_names=LABELS,
        ))

    if finetuned_results:
        print("\nFINE-TUNED (TinyLlama + QLoRA)")
        print(f"  Accuracy: {finetuned_results['accuracy']:.1%}")
        # YOUR CODE HERE — add cost and latency comparison

comparison_report(baseline_results)

## Step 6 · Analysis and Decision (Stretch)

Answer these questions in the cell below based on your results:

In [ ]:
# YOUR ANALYSIS HERE (write as a string or use markdown)
analysis = """
1. Which model performed better on accuracy? By how much?
   Answer: ...

2. At what monthly request volume would fine-tuning break even on cost vs API calls?
   Assume fine-tuning cost = $50 (compute), inference cost = $0.001/request (local GPU).
   Answer: ...

3. For what types of classification tasks would fine-tuning be clearly superior to prompting?
   Answer: ...

4. What would you need to improve if you had 10x more training data?
   Answer: ...
"""
print(analysis)